# Choquistic regression workflow

Choquistic regression models `P(Y=1 | x) = sigmoid(gamma * (C_nu(x) - beta))`. It is a probabilistic classifier fitted by constrained maximum likelihood.


In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

from capacities_ml_fin.base.interpretation import pairwise_interaction_matrix, pairwise_interactions, shapley_indices
from capacities_ml_fin.ml.model_selection import capacity_parameter_grid
from capacities_ml_fin.ml.models import ChoquisticRegression
from capacities_ml_fin.ml.optimization import L1Penalty, Solver
from capacities_ml_fin.ml.preprocessing import CapacityNormalizer

rng = np.random.default_rng(31)


## 1. Probabilistic binary data

The event probability is generated from a non-additive latent utility. `volatility` is oriented as a cost during preprocessing.


In [3]:
n = 120
X = pd.DataFrame(
    {
        "profitability": rng.uniform(5.0, 25.0, n),
        "liquidity": rng.uniform(0.8, 2.5, n),
        "volatility": rng.uniform(0.10, 0.60, n),
    }
)
oriented = np.column_stack(
    (
        (X["profitability"] - 5.0) / 20.0,
        (X["liquidity"] - 0.8) / 1.7,
        (0.60 - X["volatility"]) / 0.50,
    )
)
utility = (
    0.05 * oriented[:, 0]
    + 0.05 * oriented[:, 1]
    + 0.05 * oriented[:, 2]
    + 0.65 * np.minimum(oriented[:, 0], oriented[:, 1])
    + 0.10 * np.minimum(oriented[:, 0], oriented[:, 2])
    + 0.10 * np.minimum(oriented[:, 1], oriented[:, 2])
)
true_probability = expit(7.0 * (utility - 0.55))
y = rng.binomial(1, true_probability)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=7
)
display(X.head())
print("Class counts:", np.bincount(y))


,profitability,liquidity,volatility
0,23.063436,1.983859,0.432587
1,6.353565,0.898840,0.301093
2,18.454616,1.046441,0.313969
3,14.450894,1.459369,0.231208
4,18.548556,1.539984,0.569599


Class counts: [90 30]


## 2. Preprocess and select capacity order

ROC-AUC selects the ranking model. The normalizer is refitted within every cross-validation fold.


In [4]:
pipeline = Pipeline(
    [
        ("normalize", CapacityNormalizer(cost_features=["volatility"])),
        (
            "model",
            ChoquisticRegression(
                solver=Solver.SCIPY,
                solver_options={"options": {"maxiter": 1500, "ftol": 1e-10}},
                class_weight="balanced",
            ),
        ),
    ]
).set_output(transform="pandas")
search = GridSearchCV(
    pipeline,
    capacity_parameter_grid(parameter_name="model__sparsity", orders=(1, 2, 3)),
    scoring="roc_auc",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=4),
    n_jobs=1,
)
search.fit(X_train, y_train)

selection_results = pd.DataFrame(search.cv_results_)
selection_results["mean_validation_roc_auc"] = selection_results["mean_test_score"]
display(
    selection_results[
        ["param_model__sparsity", "mean_validation_roc_auc", "std_test_score"]
    ]
)
print("Selected capacity:", search.best_params_["model__sparsity"])


,param_model__sparsity,mean_validation_roc_auc,std_test_score
0,"KAdditivity(order=1, shape=<CapacityShape.GENE...",0.758435,0.060007
1,"KAdditivity(order=2, shape=<CapacityShape.GENE...",0.810677,0.082232
2,"KAdditivity(order=3, shape=<CapacityShape.GENE...",0.799137,0.078620


Selected capacity: KAdditivity(order=2, shape=<CapacityShape.GENERAL: 'general'>)


## 3. Regularized final model

The L1 penalty shrinks interaction Möbius coefficients while retaining singleton effects.


In [5]:
best_sparsity = search.best_params_["model__sparsity"]
compilation = best_sparsity.compile(X.shape[1])
interaction_positions = np.array(
    [i for i, mask in enumerate(compilation.bundle.parameter_masks) if mask.bit_count() >= 2],
    dtype=int,
)
penalty = L1Penalty(weight=0.01, selection=interaction_positions)

final_pipeline = Pipeline(
    [
        ("normalize", CapacityNormalizer(cost_features=["volatility"])),
        (
            "model",
            ChoquisticRegression(
                sparsity=best_sparsity,
                penalty=penalty,
                solver=Solver.SCIPY,
                solver_options={"options": {"maxiter": 2000, "ftol": 1e-12}},
                class_weight="balanced",
            ),
        ),
    ]
).set_output(transform="pandas").fit(X_train, y_train)


## 4. Probabilistic evaluation

ROC-AUC and PR-AUC are better when higher. Log loss and Brier score are better when lower. A classical logistic regression receives the same normalized inputs and provides the additive probabilistic benchmark. Hard-class metrics use the probability threshold of 0.5.


In [6]:
classical_pipeline = Pipeline(
    [
        ("normalize", CapacityNormalizer(cost_features=["volatility"])),
        (
            "model",
            LogisticRegression(class_weight="balanced", max_iter=2000),
        ),
    ]
).set_output(transform="pandas").fit(X_train, y_train)

fitted_model = final_pipeline.named_steps["model"]
probability = final_pipeline.predict_proba(X_test)[:, 1]
prediction = final_pipeline.predict(X_test)
logistic_probability = classical_pipeline.predict_proba(X_test)[:, 1]
logistic_prediction = classical_pipeline.predict(X_test)
baseline_probability = np.full(y_test.shape, np.mean(y_train), dtype=float)
baseline_prediction = (baseline_probability >= 0.5).astype(int)

probabilistic_metrics = pd.DataFrame(
    {
        "Choquistic regression": {
            "ROC-AUC": roc_auc_score(y_test, probability),
            "PR-AUC": average_precision_score(y_test, probability),
            "log loss": log_loss(y_test, probability),
            "Brier score": brier_score_loss(y_test, probability),
        },
        "logistic regression": {
            "ROC-AUC": roc_auc_score(y_test, logistic_probability),
            "PR-AUC": average_precision_score(y_test, logistic_probability),
            "log loss": log_loss(y_test, logistic_probability),
            "Brier score": brier_score_loss(y_test, logistic_probability),
        },
        "prevalence baseline": {
            "ROC-AUC": roc_auc_score(y_test, baseline_probability),
            "PR-AUC": average_precision_score(y_test, baseline_probability),
            "log loss": log_loss(y_test, baseline_probability),
            "Brier score": brier_score_loss(y_test, baseline_probability),
        },
    }
)
def hard_classification_metrics(observed, predicted):
    return {
        "accuracy": accuracy_score(observed, predicted),
        "balanced_accuracy": balanced_accuracy_score(observed, predicted),
        "precision": precision_score(observed, predicted, zero_division=0),
        "recall": recall_score(observed, predicted, zero_division=0),
        "F1": f1_score(observed, predicted, zero_division=0),
    }

classification_metrics = pd.DataFrame(
    {
        "Choquistic regression": hard_classification_metrics(y_test, prediction),
        "logistic regression": hard_classification_metrics(
            y_test, logistic_prediction
        ),
    }
)

display(probabilistic_metrics)
display(classification_metrics)
print(f"Choquistic beta: {fitted_model.beta_:.6f}")
print(f"Choquistic gamma: {fitted_model.gamma_:.6f}")
display(pd.DataFrame(confusion_matrix(y_test, prediction), index=["true 0", "true 1"], columns=["pred 0", "pred 1"]))
display(pd.DataFrame({"observed": y_test[:8], "probability": probability[:8], "predicted": prediction[:8]}))


,Choquistic regression,logistic regression,prevalence baseline
ROC-AUC,0.857955,0.823864,0.500000
PR-AUC,0.618002,0.572917,0.266667
log loss,0.592118,0.598491,0.581226
Brier score,0.194488,0.206477,0.196049


,Choquistic regression,logistic regression
accuracy,0.700000,0.700000
balanced_accuracy,0.795455,0.795455
precision,0.470588,0.470588
recall,1.000000,1.000000
F1,0.640000,0.640000


Choquistic beta: 0.466149
Choquistic gamma: 9.910682


,pred 0,pred 1
true 0,13,9
true 1,0,8


,observed,probability,predicted
0,0,0.163522,0
1,0,0.520883,1
2,0,0.475290,0
3,1,0.728197,1
4,1,0.903977,1
5,1,0.600676,1
6,0,0.042134,0
7,0,0.669110,1


## 5. Interpret latent utility

Shapley and interaction indices explain the learned Choquet utility before the logistic link.


In [7]:
display(pd.Series(shapley_indices(fitted_model.capacity_), name="Shapley importance"))
display(pd.Series(pairwise_interactions(fitted_model.capacity_), name="pairwise interaction"))
display(
    pd.DataFrame(
        pairwise_interaction_matrix(fitted_model.capacity_),
        index=X.columns,
        columns=X.columns,
    )
)


profitability    0.381769
liquidity        0.345934
volatility       0.272297
Name: Shapley importance, dtype: float64

profitability  liquidity     0.593355
               volatility    0.170184
liquidity      volatility   -0.098514
Name: pairwise interaction, dtype: float64

,profitability,liquidity,volatility
profitability,0.000000,0.593355,0.170184
liquidity,0.593355,0.000000,-0.098514
volatility,0.170184,-0.098514,0.000000


## 6. Probabilities for new observations


In [8]:
X_new = pd.DataFrame(
    {
        "profitability": [8.0, 17.0, 24.0],
        "liquidity": [1.0, 1.7, 2.3],
        "volatility": [0.55, 0.32, 0.14],
    }
)
display(
    X_new.assign(
        positive_probability=final_pipeline.predict_proba(X_new)[:, 1],
        predicted_class=final_pipeline.predict(X_new),
    )
)


,profitability,liquidity,volatility,positive_probability,predicted_class
0,8.0,1.0,0.55,0.028477,0
1,17.0,1.7,0.32,0.693014,1
2,24.0,2.3,0.14,0.988177,1
